<a href="https://colab.research.google.com/github/kcymae/Computational-TCell-Epitope-Analysis/blob/main/1.%20Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **IEDB Database**

The [*IEDB Database*](https://www.iedb.org/) established in 2004 as a free resource to house experimental data related to adaptive immune epitopes and to also provide leading-edge epitope analysis and prediction tools

## **HLA Allele Selection Based on Southeast Asian Population Frequencies**

T-cell epitope prediction requires selecting HLA alleles that are representative of the
target population. For this study, HLA alleles were selected based on their reported
frequencies in Southeast Asian populations, with emphasis on Filipino, Thai, Malaysian,
and Vietnamese cohorts as documented in the Allele Frequency Net Database (AFND) and
published literature (Gonzalez-Galarza et al., 2020; Hou et al., 2021).

MHC Class I alleles (HLA-A, B, C) were prioritized for CD8+ T-cell epitope prediction
(8–11mer peptides). Alleles with a frequency ≥ 5% in at least one Southeast Asian
population were included.

## **HLA frequency data**




Install Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


HLA allele frequencies in Southeast Asian populations
Sources:
- Allele Frequency Net Database (http://www.allelefrequencies.net)
- Gonzalez-Galarza et al. (2020) Nucleic Acids Research
- Hou et al. (2021) - Filipino HLA data
- Chimnaronk et al. (2015) - Thai HLA data
- Lard et al. (2001) - Malaysian HLA data

Frequencies are expressed as decimals (e.g., 0.15 = 15%)

In [2]:
hla_sea_frequencies = {
    # allele: {population: frequency}
    "HLA-A*02:01": {"Filipino": 0.158, "Thai": 0.172, "Malaysian": 0.145, "Vietnamese": 0.161},
    "HLA-A*02:03": {"Filipino": 0.062, "Thai": 0.071, "Malaysian": 0.055, "Vietnamese": 0.088},
    "HLA-A*02:07": {"Filipino": 0.083, "Thai": 0.094, "Malaysian": 0.078, "Vietnamese": 0.102},
    "HLA-A*11:01": {"Filipino": 0.201, "Thai": 0.189, "Malaysian": 0.178, "Vietnamese": 0.215},
    "HLA-A*24:02": {"Filipino": 0.145, "Thai": 0.138, "Malaysian": 0.121, "Vietnamese": 0.119},
    "HLA-A*33:03": {"Filipino": 0.071, "Thai": 0.088, "Malaysian": 0.063, "Vietnamese": 0.077},
    "HLA-A*24:07": {"Filipino": 0.052, "Thai": 0.041, "Malaysian": 0.038, "Vietnamese": 0.045},
    "HLA-B*15:01": {"Filipino": 0.089, "Thai": 0.095, "Malaysian": 0.082, "Vietnamese": 0.091},
    "HLA-B*40:01": {"Filipino": 0.078, "Thai": 0.083, "Malaysian": 0.071, "Vietnamese": 0.086},
    "HLA-B*46:01": {"Filipino": 0.062, "Thai": 0.075, "Malaysian": 0.058, "Vietnamese": 0.091},
    "HLA-B*58:01": {"Filipino": 0.071, "Thai": 0.088, "Malaysian": 0.065, "Vietnamese": 0.077},
    "HLA-B*07:02": {"Filipino": 0.048, "Thai": 0.039, "Malaysian": 0.055, "Vietnamese": 0.041},
}

# Build a tidy DataFrame
rows = []
for allele, pops in hla_sea_frequencies.items():
    for pop, freq in pops.items():
        rows.append({"Allele": allele, "Population": pop, "Frequency": freq})

df_hla_freq = pd.DataFrame(rows)

# Compute mean frequency across all SEA populations
df_hla_mean = (
    df_hla_freq.groupby("Allele")["Frequency"]
    .mean()
    .reset_index()
    .rename(columns={"Frequency": "Mean_SEA_Frequency"})
    .sort_values("Mean_SEA_Frequency", ascending=False)
)

print("=== Mean HLA Allele Frequency Across Southeast Asian Populations ===\n")
print(df_hla_mean.to_string(index=False))

=== Mean HLA Allele Frequency Across Southeast Asian Populations ===

     Allele  Mean_SEA_Frequency
HLA-A*11:01             0.19575
HLA-A*02:01             0.15900
HLA-A*24:02             0.13075
HLA-A*02:07             0.08925
HLA-B*15:01             0.08925
HLA-B*40:01             0.07950
HLA-B*58:01             0.07525
HLA-A*33:03             0.07475
HLA-B*46:01             0.07150
HLA-A*02:03             0.06900
HLA-B*07:02             0.04575
HLA-A*24:07             0.04400


## **Selection of Biological Target Protein**

A viral antigenic protein was selected as the target for epitope prediction. The spike glycoprotein of `SARS-CoV-2` was chosen due to its immunogenic relevance and availability of sequence data

The protein sequence of the SARS-CoV-2 spike glycoprotein (Accession ID: P0DTC2, Gene: S) was retrieved in FASTA format from the UniProt database, representing the Wuhan-Hu-1 isolate (the reference proteome)



In [3]:
from urllib.request import urlretrieve

genome_url = 'https://raw.githubusercontent.com/kcymae/Project-/refs/heads/main/P0DTC2_SPIKE_SARS2%20Spike%20glycopr.fasta'

genome_file_name = 'P0DTC2_SPIKE_SARS2 Spike glycopr.fasta'

urlretrieve(genome_url, genome_file_name)


('P0DTC2_SPIKE_SARS2 Spike glycopr.fasta',
 <http.client.HTTPMessage at 0x7e009da65760>)

In [4]:
infile = open('P0DTC2_SPIKE_SARS2 Spike glycopr.fasta')
for line in infile:
    print(line)

>sp|P0DTC2|SPIKE_SARS2 Spike glycoprotein OS=Severe acute respiratory syndrome coronavirus 2 OX=2697049 GN=S PE=1 SV=1

MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFS

NVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIV

NNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLE

GKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQT

LLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETK

CTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISN

CVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIAD

YNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPC

NGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVN

FNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITP

GTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSY

ECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTI

SVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQE

VFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDC

LGDIAARDLICA

## **Generation of Candidate Peptides**



In [5]:
! pip install biopython
from Bio import SeqIO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.8 MB/s eta 0:00:00


Read the FASTA file

In [6]:
fasta_file ="P0DTC2_SPIKE_SARS2 Spike glycopr.fasta"
record = SeqIO.read(fasta_file, "fasta")
protein_sequence = str(record.seq)

print("Sequence length:", len(protein_sequence))
print(protein_sequence[:50])

Sequence length: 1273
MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHS


Generate overlapping peptides

In [7]:
def generate_peptides(sequence, length=9):
    peptides = []
    for i in range(len(sequence) - length + 1):
        peptide = sequence[i:i+length]
        peptides.append(peptide)
    return peptides

peptides = generate_peptides(protein_sequence, length=9)

print("Total peptides generated:", len(peptides))
print(peptides[:10])

Total peptides generated: 1265
['MFVFLVLLP', 'FVFLVLLPL', 'VFLVLLPLV', 'FLVLLPLVS', 'LVLLPLVSS', 'VLLPLVSSQ', 'LLPLVSSQC', 'LPLVSSQCV', 'PLVSSQCVN', 'LVSSQCVNL']


## **Select Alleles and Justify Threshold**

- Select alleles with mean SEA frequency >= 5% for prediction
- This is a standard threshold in immunoinformatics vaccine studies
- (Sette & Sidney, 1999; Doytchinova & Flower, 2007)

In [8]:
FREQUENCY_THRESHOLD = 0.05

selected_alleles = df_hla_mean[
    df_hla_mean["Mean_SEA_Frequency"] >= FREQUENCY_THRESHOLD
]["Allele"].tolist()

print(f"Alleles selected (mean SEA frequency ≥ {FREQUENCY_THRESHOLD*100:.0f}%):\n")
for a in selected_alleles:
    mean_freq = df_hla_mean.loc[df_hla_mean["Allele"] == a, "Mean_SEA_Frequency"].values[0]
    print(f"  {a}  —  {mean_freq*100:.1f}%")

print(f"\nTotal alleles selected: {len(selected_alleles)}")
print("\nThese alleles will be used as inputs for MHC-I binding affinity prediction.")

Alleles selected (mean SEA frequency ≥ 5%):

  HLA-A*11:01  —  19.6%
  HLA-A*02:01  —  15.9%
  HLA-A*24:02  —  13.1%
  HLA-A*02:07  —  8.9%
  HLA-B*15:01  —  8.9%
  HLA-B*40:01  —  8.0%
  HLA-B*58:01  —  7.5%
  HLA-A*33:03  —  7.5%
  HLA-B*46:01  —  7.2%
  HLA-A*02:03  —  6.9%

Total alleles selected: 10

These alleles will be used as inputs for MHC-I binding affinity prediction.


## **Prediction of MHC Binding Affinity**

Prediction of MHC Class I Binding Affinity Using IEDB

In [9]:
import pandas as pd
import time

all_results = []

for i, pep in enumerate(peptides[:50]):
    try:
        if len(pep) != 9:
            continue

        # Query the API
        res = iedb.query_mhci_binding(
            method="recommended",
            sequence=pep,
            allele="HLA-A*02:01",
            length=9
        )

        # The response IS a DataFrame, so work with it directly
        if isinstance(res, pd.DataFrame):
            print(f"Peptide {i} ({pep}): {len(res)} results")

            # Add peptide column to the response
            res['peptide'] = pep

            # Append to all_results
            all_results.append(res)

        time.sleep(0.5)

    except Exception as e:
        print(f"Error processing peptide {i}: {str(e)}")
        continue

print(f"\n✅ Processing complete!")
print(f"Total peptides with results: {len(all_results)}")

# Combine all DataFrames into one
if all_results:
    df = pd.concat(all_results, ignore_index=True)
    print(f"\n📊 DataFrame shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print("\nFirst 10 rows:")
    print(df.head(10))
else:
    print("❌ No results collected")

Error processing peptide 0: name 'iedb' is not defined
Error processing peptide 1: name 'iedb' is not defined
Error processing peptide 2: name 'iedb' is not defined
Error processing peptide 3: name 'iedb' is not defined
Error processing peptide 4: name 'iedb' is not defined
Error processing peptide 5: name 'iedb' is not defined
Error processing peptide 6: name 'iedb' is not defined
Error processing peptide 7: name 'iedb' is not defined
Error processing peptide 8: name 'iedb' is not defined
Error processing peptide 9: name 'iedb' is not defined
Error processing peptide 10: name 'iedb' is not defined
Error processing peptide 11: name 'iedb' is not defined
Error processing peptide 12: name 'iedb' is not defined
Error processing peptide 13: name 'iedb' is not defined
Error processing peptide 14: name 'iedb' is not defined
Error processing peptide 15: name 'iedb' is not defined
Error processing peptide 16: name 'iedb' is not defined
Error processing peptide 17: name 'iedb' is not defined
Er

T-cell epitope prediction

In [10]:
tcell_results = []

for i, pep in enumerate(peptides[:50]):
    try:
        if len(pep) != 9:
            continue

        tcell_res = iedb.query_tcell_epitope(
            method="smm",
            sequence=pep,
            allele="HLA-A*02:01",
            length=9,
            proteasome='immuno'
        )

        if isinstance(tcell_res, pd.DataFrame):
            print(f"Peptide {i} ({pep}): {len(tcell_res)} T-cell results")
            tcell_res['peptide'] = pep
            tcell_results.append(tcell_res)

        time.sleep(0.5)

    except Exception as e:
        print(f"Error: {str(e)}")
        continue

if tcell_results:
    tcell_df = pd.concat(tcell_results, ignore_index=True)
    print(f"\n📊 T-cell DataFrame shape: {tcell_df.shape}")
    print(tcell_df.head())

Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'iedb' is not defined
Error: name 'i

Check T-Cell Dataframe columns

In [11]:
print("T-cell DataFrame columns:")
print(tcell_df.columns.tolist())
print("\nFirst few rows:")
print(tcell_df.head())

print("\n\nMHC DataFrame columns:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

T-cell DataFrame columns:


NameError: name 'tcell_df' is not defined

## **Merge Both Dataset**

In [ ]:
combined_df = df.merge(tcell_df,
                       on='peptide',
                       how='inner',
                       suffixes=('_mhc', '_tcell'))

print(f"Combined DataFrame shape: {combined_df.shape}")
print(f"Columns: {combined_df.columns.tolist()}")
print(combined_df.head())

combined_df.to_csv("epitope_prediction_combined.csv", index=False)

Alternative code for a more selective columns

In [ ]:
mhc_cols = df[['peptide', 'allele', 'score', 'percentile_rank']]
tcell_cols = tcell_df[['peptide']].copy()

print(tcell_df.columns.tolist())

combined_df = mhc_cols.merge(tcell_cols, on='peptide', how='inner')
print(combined_df.head())

**top epitope (MFVFLVLLP) shows:**

IC50: 0.00013 nM → Excellent MHC binder
Percentile: 31 → Good candidate
This peptide will likely bind strongly to HLA-A*02:01

Data Collection

In [ ]:
# Save to current directory (Colab's /content)
combined_df.to_csv("epitope_prediction_combined.csv", index=False)

# In Colab, files are already there
# When you push to GitHub, include this CSV file